# M5 — Phân tích kết quả GAN V2 từ ZIP / output cũ (CPU)

**Không train, không chạy lại inference và không cần GPU.** Notebook tìm file CSV/JSON/Markdown trong ZIP hoặc thư mục output; chỉ tính chỉ số trên dữ liệu thật tìm thấy.

**Cách dùng:** (1) upload ZIP vào Kaggle `/kaggle/input/` hoặc đặt ZIP cạnh notebook trên VS Code; (2) chạy tất cả cell; (3) xem danh sách file và thông báo thiếu dữ liệu. Nếu auto-detect không đúng, chỉnh `MANUAL_METRICS` / `MANUAL_PREDICTIONS` ở cell cấu hình.

> Không tải checkpoint `.h5/.keras/.ckpt/.pt/.pth`; không giải nén file thực thi; giới hạn ZIP tránh file quá lớn. Nếu tệp chỉ chứa summary `mean ± std`, không thể suy ra giá trị từng seed hoặc F1 từng lớp.

In [ ]:
from pathlib import Path
import os, re, json, zipfile, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Có thể đặt đường dẫn tới ZIP hoặc thư mục cụ thể. None = tự tìm.
INPUT_PATH = None  # ví dụ '/kaggle/input/my-output/results.zip' hoặc r'D:\GAN\output.zip'
MANUAL_METRICS = None      # ví dụ '/kaggle/working/m5_input/metrics.csv'
MANUAL_PREDICTIONS = None  # ví dụ '/kaggle/working/m5_input/predictions.csv'
OUTPUT_DIR = Path('/kaggle/working/m5_v2_analysis') if Path('/kaggle/working').exists() else Path.cwd() / 'm5_v2_analysis'
EXTRACT_DIR = OUTPUT_DIR / 'extracted'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ALIASES = {'learnedlocal':'learned_local','learned_local':'learned_local','v2':'learned_local',
           'randomlocal':'random_local','random_local':'random_local',
           'matchedros':'matched_ros','matched_ros':'matched_ros','ros':'matched_ros',
           'originalgan':'original_gan','original_gan':'original_gan','original':'original_gan'}
METRIC_NAMES = ['oa','aa','macro_f1','gm','kappa']
print('CPU analysis; output:', OUTPUT_DIR)


## 1. Tìm ZIP, giải nén an toàn và kiểm kê file

Nếu có nhiều ZIP, đặt `INPUT_PATH` để chỉ định đúng ZIP kết quả M5. Notebook **không** đoán nội dung từ tên checkpoint.

In [ ]:
roots = []
if INPUT_PATH:
    p = Path(INPUT_PATH).expanduser()
    if not p.exists(): raise FileNotFoundError(f'INPUT_PATH không tồn tại: {p}')
    roots = [p]
else:
    roots = [Path.cwd()]
    if Path('/kaggle/input').exists(): roots.append(Path('/kaggle/input'))
    if Path('/kaggle/working').exists(): roots.append(Path('/kaggle/working'))
roots = list(dict.fromkeys(roots))

# Giới hạn quét để tránh đọc nhầm dataset hoặc checkpoint khổng lồ.
def walk_limited(root, max_files=12000):
    if root.is_file():
        yield root; return
    seen = 0
    for base, dirs, fs in os.walk(root):
        dirs[:] = [d for d in dirs if d not in {'.git','.venv','node_modules','site-packages','m5_v2_analysis'}]
        for fn in fs:
            yield Path(base) / fn
            seen += 1
            if seen >= max_files:
                print('Giới hạn quét 12.000 file tại', root)
                return

files = sorted(set(p for root in roots for p in walk_limited(root)), key=str)
zip_paths = [p for p in files if p.suffix.lower()=='.zip' and p.stat().st_size < 3_000_000_000]
print('ZIP tìm thấy:', len(zip_paths))
for p in zip_paths[:30]: print(' ZIP:',p)

MAX_ZIP_TOTAL = 500 * 1024**2  # <= 500 MB file được chọn
MAX_ZIP_SINGLE = 100 * 1024**2
ALLOWED_SUFFIX = {'.csv','.tsv','.txt','.md','.json','.jsonl','.npy','.npz'}
def safe_extract(zpath, dest):
    dest.mkdir(parents=True, exist_ok=True)
    total = 0; kept = 0
    try:
        with zipfile.ZipFile(zpath) as z:
            for item in z.infolist():
                rel = Path(item.filename)
                if item.is_dir() or rel.is_absolute() or '..' in rel.parts or rel.suffix.lower() not in ALLOWED_SUFFIX:
                    continue
                if item.file_size > MAX_ZIP_SINGLE or total + item.file_size > MAX_ZIP_TOTAL:
                    continue
                target = dest.joinpath(*rel.parts)
                target.parent.mkdir(parents=True, exist_ok=True)
                with z.open(item) as src, open(target,'wb') as dst: shutil.copyfileobj(src,dst)
                total += item.file_size; kept += 1
        print(f'{zpath.name}: lấy {kept} file dữ liệu; {total/1024**2:.1f} MiB')
    except (zipfile.BadZipFile, RuntimeError, OSError) as e:
        print('Không thể đọc ZIP:', zpath, str(e))

for i,zpath in enumerate(zip_paths[:15]):
    safe_extract(zpath, EXTRACT_DIR / f'zip_{i:02d}')

all_data = [p for p in files if p.suffix.lower() in ALLOWED_SUFFIX and p.stat().st_size < MAX_ZIP_SINGLE]
if EXTRACT_DIR.exists(): all_data += [p for p in EXTRACT_DIR.rglob('*') if p.is_file()]
all_data = sorted(set(all_data),key=str)
manifest = pd.DataFrame([{'path':str(p),'name':p.name,'size_KiB':round(p.stat().st_size/1024,1)} for p in all_data])
display(manifest.head(100))
manifest.to_csv(OUTPUT_DIR/'file_manifest.csv', index=False)


## 2. Nhận diện file metrics theo seed

Hỗ trợ CSV/TSV, JSON dạng danh sách bản ghi, Markdown table (`RESULTS.md`) **nếu** bảng có cột `seed` và `branch` rõ ràng. Không suy giá trị từng seed từ bảng chỉ chứa trung bình ± độ lệch chuẩn.

In [ ]:
def slug(x):
    return re.sub(r'[^a-z0-9]+','_',str(x).strip().lower()).strip('_')
def normalize_branch(s):
    k=slug(s)
    return ALIASES.get(k, k)
def canonical_columns(df):
    a = df.copy()
    colmap = {'method':'branch','model':'branch','variant':'branch','arm':'branch',
       'run_seed':'seed','random_seed':'seed','overall_accuracy':'oa','accuracy':'oa',
       'average_accuracy':'aa','macro_f1_score':'macro_f1','f1_macro':'macro_f1',
       'macrof1':'macro_f1','macro_f1':'macro_f1','g_mean':'gm','gmean':'gm',
       'geometric_mean':'gm','cohen_kappa':'kappa'}
    a.columns = [colmap.get(slug(c),slug(c)) for c in a.columns]
    return a

def read_table(path):
    suf=path.suffix.lower()
    try:
        if suf in ('.csv','.tsv'):
            return pd.read_csv(path,sep='\t' if suf=='.tsv' else None,engine='python')
        if suf=='.json':
            obj=json.loads(path.read_text(encoding='utf-8-sig'))
            if isinstance(obj,list): return pd.json_normalize(obj)
            if isinstance(obj,dict):
                for key in ('rows','records','results','metrics','predictions','data'):
                    if isinstance(obj.get(key),list): return pd.json_normalize(obj[key])
        if suf in ('.md','.txt') and path.stat().st_size < 3_000_000:
            lines=path.read_text(encoding='utf-8-sig',errors='replace').splitlines()
            for i,line in enumerate(lines):
                if '|' not in line or 'seed' not in line.lower(): continue
                h=[v.strip() for v in line.strip().strip('|').split('|')]
                if len(h)<3: continue
                j=i+1
                if j<len(lines) and re.fullmatch(r'[\s|:\-]+',lines[j]): j+=1
                rows=[]
                while j<len(lines) and '|' in lines[j]:
                    v=[x.strip() for x in lines[j].strip().strip('|').split('|')]
                    if len(v)==len(h): rows.append(v)
                    j+=1
                if rows: return pd.DataFrame(rows,columns=h)
    except Exception as e:
        print('Bỏ qua file không đọc được',path.name,':',str(e)[:160])
    return None

def validate_metrics(raw):
    if raw is None:return None
    d=canonical_columns(raw)
    if not {'branch','seed'} <= set(d.columns):return None
    metric_cols=[c for c in METRIC_NAMES if c in d.columns]
    if not metric_cols:return None
    d['seed']=pd.to_numeric(d['seed'],errors='coerce')
    d['branch']=d['branch'].map(normalize_branch)
    for c in metric_cols:
        d[c]=pd.to_numeric(d[c].astype(str).str.replace('%','',regex=False).str.strip(),errors='coerce')
    d=d.dropna(subset=['seed'])
    d=d.dropna(subset=metric_cols,how='all')
    if d.empty:return None
    d['seed']=d['seed'].astype(int)
    return d[['branch','seed']+metric_cols].copy()

metric_candidates=[]
search_paths=([Path(MANUAL_METRICS)] if MANUAL_METRICS else all_data)
for p in search_paths:
    if p.suffix.lower() not in {'.csv','.tsv','.json','.md','.txt'}:continue
    d=validate_metrics(read_table(p))
    if d is not None:
        metric_candidates.append((p,d))
print('Bảng metrics phù hợp:',len(metric_candidates))
for i,(p,d) in enumerate(metric_candidates): print(i, p,'=>',d.shape, 'nhánh:',sorted(d.branch.unique()))
# Ưu tiên file chứa nhiều nhánh/seed và tên metrics/results.
def metric_score(item):
    p,d=item
    return (len(d.branch.unique())*100 + len(d.seed.unique())*10 + len(d),int(bool(re.search('metrics|results|m5',p.name,re.I))))
metric_candidates.sort(key=metric_score,reverse=True)
metrics = metric_candidates[0][1] if metric_candidates else None
if metrics is None: print('THIẾU bảng metrics theo seed: không thể phân tích seed 43 từ dữ liệu tổng hợp.')
else:
    print('Đang dùng:',metric_candidates[0][0]);display(metrics.head(30))
    if metrics.duplicated(['branch','seed']).any():
        print('CẢNH BÁO: trùng branch/seed, không tự gộp. Kiểm tra đầu vào trước khi phân tích.')


## 3. So sánh seed, đặc biệt seed 43 (không train)

Chênh lệch có ghép cặp theo seed. Với chỉ 3 seed, **không kết luận ý nghĩa thống kê**.

In [ ]:
summary=None; paired=None
if metrics is not None and not metrics.duplicated(['branch','seed']).any():
    value_cols=[x for x in METRIC_NAMES if x in metrics.columns]
    summary=metrics.groupby('branch')[value_cols].agg(['mean','std','count']).round(4)
    display(summary)
    target=metrics.loc[metrics.branch=='learned_local'].set_index('seed')
    pieces=[]
    for other in ('original_gan','matched_ros','random_local'):
        comp=metrics.loc[metrics.branch==other].set_index('seed')
        common=target.index.intersection(comp.index)
        if len(common)==0:continue
        tmp=(target.loc[common,value_cols]-comp.loc[common,value_cols]).reset_index()
        tmp.insert(1,'versus',other)
        pieces.append(tmp)
    paired=pd.concat(pieces,ignore_index=True) if pieces else pd.DataFrame()
    print('Learned-local trừ đối chứng, theo cùng seed:')
    display(paired)
    print('Riêng seed 43:')
    display(metrics[metrics.seed==43].sort_values('branch'))
    if not paired.empty:
        display(paired[paired.seed==43])
else:print('Chưa có metrics hợp lệ theo từng seed.')


## 4. Nhận diện predictions và đánh giá từng lớp

Hỗ trợ CSV/TSV/JSON có các cột `branch`, `seed`, `y_true`, `y_pred` (hoặc `true`, `pred`, `label`, `prediction`). Nếu nhãn trong `.npy/.npz` hoặc tên cột khác, notebook **liệt kê file** nhưng không tự đoán cặp nhãn nhằm tránh phân tích sai. Có thể bổ sung một cell chuyển đổi phù hợp sau khi xem file thực tế.

In [ ]:
def validate_predictions(raw):
    if raw is None:return None
    d=canonical_columns(raw)
    ren={'true':'y_true','ground_truth':'y_true','target':'y_true','label':'y_true',
         'pred':'y_pred','prediction':'y_pred','predicted':'y_pred','predicted_label':'y_pred',
         'class':'y_true'}
    d=d.rename(columns={c:ren[c] for c in d.columns if c in ren and ren[c] not in d.columns})
    if not {'branch','seed','y_true','y_pred'}<=set(d.columns):return None
    d=d.copy()
    for c in ('seed','y_true','y_pred'):d[c]=pd.to_numeric(d[c],errors='coerce')
    d=d.dropna(subset=['seed','y_true','y_pred'])
    if d.empty:return None
    d['branch']=d['branch'].map(normalize_branch)
    for c in ('seed','y_true','y_pred'):d[c]=d[c].astype(int)
    return d

pred_candidates=[]
search_paths=([Path(MANUAL_PREDICTIONS)] if MANUAL_PREDICTIONS else all_data)
for p in search_paths:
    if p.suffix.lower() not in {'.csv','.tsv','.json'}:continue
    d=validate_predictions(read_table(p))
    if d is not None:pred_candidates.append((p,d))
print('Bảng predictions phù hợp:',len(pred_candidates))
for i,(p,d) in enumerate(pred_candidates): print(i,p,'=>',d.shape)
pred_candidates.sort(key=lambda q:len(q[1]),reverse=True)
preds=pred_candidates[0][1] if pred_candidates else None
if preds is None:
    print('Không có predictions dạng bảng chuẩn. Không tính F1 từng lớp bằng cách suy đoán.')
    print('Các file NPY/NPZ có thể cần đọc riêng:',*[str(p) for p in all_data if p.suffix.lower() in ('.npy','.npz')][:25],sep='\n - ')
else:
    print('Đang dùng:',pred_candidates[0][0]);display(preds.head())


In [ ]:
per_class=None; per_seed_from_preds=None; deltas=None
if preds is not None:
    from sklearn.metrics import precision_recall_fscore_support, confusion_matrix, accuracy_score, cohen_kappa_score, recall_score, f1_score
    labels=sorted(set(preds.y_true.unique())|set(preds.y_pred.unique()))
    print('Nhãn xuất hiện:',labels)
    if 0 in labels:
        print('CẢNH BÁO: nhãn 0 có thể là background của Indian Pines; kiểm tra và loại đúng theo protocol test trước khi báo cáo.')
    rows=[];seed_rows=[]
    for (branch,seed),d in preds.groupby(['branch','seed']):
        y=d.y_true.to_numpy();yp=d.y_pred.to_numpy()
        pr,re,fs,sup=precision_recall_fscore_support(y,yp,labels=labels,zero_division=0)
        for cls,p,r,f,s in zip(labels,pr,re,fs,sup):
            rows.append(dict(branch=branch,seed=seed,class_id=cls,support=int(s),precision=p,recall=r,f1=f))
        seed_rows.append(dict(branch=branch,seed=seed,oa=accuracy_score(y,yp)*100,
            aa=np.mean(re)*100,macro_f1=f1_score(y,yp,labels=labels,average='macro',zero_division=0)*100,
            kappa=cohen_kappa_score(y,yp,labels=labels)))
        cm=confusion_matrix(y,yp,labels=labels)
        pd.DataFrame(cm,index=[f'true_{k}' for k in labels],columns=[f'pred_{k}' for k in labels]).to_csv(OUTPUT_DIR/f'confusion_{branch}_seed{seed}.csv')
    per_class=pd.DataFrame(rows);per_seed_from_preds=pd.DataFrame(seed_rows)
    display(per_seed_from_preds)
    compare=per_class.pivot_table(index=['seed','class_id'],columns='branch',values='f1')
    if {'learned_local','random_local'}<=set(compare.columns):
        deltas=(compare['learned_local']-compare['random_local']).rename('delta_f1_learned_minus_random').reset_index()
        print('Chênh lệch F1 từng lớp:');display(deltas.sort_values('delta_f1_learned_minus_random').head(30))
    print('Class F1 seed 43:')
    display(per_class[per_class.seed==43].pivot_table(index='class_id',columns='branch',values='f1'))
else:
    print('Bỏ qua phần per-class (không có y_true/y_pred).')


## 5. Xuất CSV và checklist kết luận

Bảng được xuất ngay cả khi chỉ có một phần dữ liệu. Gửi cả ZIP output gốc khi cần kiểm chứng nguồn số liệu.

In [ ]:
if metrics is not None:metrics.to_csv(OUTPUT_DIR/'metrics_detected.csv',index=False)
if summary is not None: summary.to_csv(OUTPUT_DIR/'metrics_summary.csv')
if paired is not None and not paired.empty:paired.to_csv(OUTPUT_DIR/'paired_deltas_by_seed.csv',index=False)
if per_class is not None:per_class.to_csv(OUTPUT_DIR/'per_class_metrics.csv',index=False)
if per_seed_from_preds is not None:per_seed_from_preds.to_csv(OUTPUT_DIR/'metrics_recomputed_from_predictions.csv',index=False)
if deltas is not None:deltas.to_csv(OUTPUT_DIR/'per_class_learned_minus_random.csv',index=False)
report={'metrics_by_seed_found':metrics is not None,
        'seed_43_found':bool(metrics is not None and (metrics.seed==43).any()),
        'predictions_found':preds is not None,
        'per_class_computed':per_class is not None,
        'source_metrics':str(metric_candidates[0][0]) if metric_candidates else None,
        'source_predictions':str(pred_candidates[0][0]) if pred_candidates else None}
(OUTPUT_DIR/'analysis_status.json').write_text(json.dumps(report,indent=2,ensure_ascii=False),encoding='utf-8')
print(json.dumps(report,indent=2,ensure_ascii=False))
print('Đã xuất:', OUTPUT_DIR)
print('Lưu ý: kết quả dùng để mô tả bằng chứng thực nghiệm, không đủ để chứng minh ý nghĩa thống kê khi chỉ có 3 seed.')
